# Neural Networks from Scratch — 3. The Same Network in PyTorch

**Notebook 3 of 4**

| # | Notebook | What you build |
|---|---|---|
| 1 | NumPy foundations | A single neuron, trained by hand |
| 2 | A network from scratch | A 2-layer network with backpropagation |
| **3** | **The same network in PyTorch** | **Tensors, autograd, `nn.Module`** |
| 4 | MNIST and a small CNN | A real image classifier |

---

## What this notebook is for

You have already written a working neural network. This notebook doesn't teach you
anything new about how networks *work* — it shows you how to stop writing the same
fifteen lines of calculus every time.

We will build **the exact same network as notebook 2**, four times over, each version
shorter than the last:

1. raw tensors with autograd — PyTorch computes the backward pass, you write the rest
2. `nn.Linear` — layers as objects
3. `nn.Sequential` + `torch.optim` — the loop shrinks to five lines
4. `nn.Module` — the style you'll see in every real codebase

Along the way we'll take PyTorch's gradients and check them against the ones we
derived by hand in notebook 2. They agree exactly. That's the moment autograd stops
being magic: it is doing precisely what you already did, just automatically.

## Installing PyTorch

CPU-only is all we need:

```
pip install torch torchvision
```

On the PyTorch website you'll be offered CUDA builds. Ignore them — everything in
these notebooks runs comfortably on a laptop CPU, and the CPU build is a much
smaller download.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

print("torch  ", torch.__version__)
print("numpy  ", np.__version__)

# Everything here is CPU-only and deliberately small.
torch.manual_seed(0)

plt.rcParams["figure.figsize"] = (6, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
plt.rcParams["figure.dpi"] = 110

> If that import failed, PyTorch isn't installed in the environment your notebook is
> running in. The usual culprit is having several Pythons on one machine. Run
> `import sys; print(sys.executable)` in a cell, and use *that* interpreter's pip:
> `!{sys.executable} -m pip install torch torchvision`.

---
# Setup: the data and the NumPy network

Same two moons as notebook 2, and a compact copy of the from-scratch network so we
have something to compare against. Nothing here is new — skim it and move on.

In [ ]:
from sklearn.datasets import make_moons

X, y = make_moons(n_samples=400, noise=0.20, random_state=3)
y = y.astype(float)
Y = y.reshape(-1, 1)          # column form, as always

print("X", X.shape, " Y", Y.shape)


def plot_decision_boundary(predict_fn, X, y, title="", ax=None):
    """Shade the plane by what predict_fn says, then scatter the real data on top."""
    if ax is None:
        _, ax = plt.subplots()
    pad = 0.5
    xx, yy = np.meshgrid(
        np.linspace(X[:, 0].min() - pad, X[:, 0].max() + pad, 300),
        np.linspace(X[:, 1].min() - pad, X[:, 1].max() + pad, 300))
    grid = np.c_[xx.ravel(), yy.ravel()]
    zz = predict_fn(grid).reshape(xx.shape)
    ax.contourf(xx, yy, zz, levels=50, cmap="coolwarm", alpha=0.7)
    ax.contour(xx, yy, zz, levels=[0.5], colors="k", linewidths=2)
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", edgecolor="k", s=35)
    ax.set_title(title)
    return ax

In [ ]:
# --- the notebook 2 network, condensed ---

def np_init(n_input, n_hidden, seed=0):
    r = np.random.default_rng(seed)
    return {"W1": r.normal(size=(n_input, n_hidden)) * np.sqrt(2.0 / n_input),
            "b1": np.zeros(n_hidden),
            "W2": r.normal(size=(n_hidden, 1)) * np.sqrt(2.0 / n_hidden),
            "b2": np.zeros(1)}


def np_forward(X, p):
    Z1 = X @ p["W1"] + p["b1"]
    A1 = np.maximum(0.0, Z1)
    Z2 = A1 @ p["W2"] + p["b2"]
    A2 = 1.0 / (1.0 + np.exp(-Z2))
    return A2, {"Z1": Z1, "A1": A1, "A2": A2}


def np_backward(X, Y, p, cache):
    n = X.shape[0]
    dZ2 = (cache["A2"] - Y) / n
    dW2 = cache["A1"].T @ dZ2
    db2 = dZ2.sum(axis=0)
    dZ1 = (dZ2 @ p["W2"].T) * (cache["Z1"] > 0)
    return {"W1": X.T @ dZ1, "b1": dZ1.sum(axis=0), "W2": dW2, "b2": db2}


np_params = np_init(2, 16, seed=0)
print("parameter shapes:", {k: v.shape for k, v in np_params.items()})

We use **16 hidden units** here rather than the 8 from notebook 2. Experiment 2 in
that notebook showed why: narrow layers are seed-dependent and sometimes fail to find
the curve at all, while 16 is reliably fine. Since this notebook is about PyTorch
mechanics rather than about capacity, we don't want random failures distracting us.

---
# 5.1 Tensors

A PyTorch **tensor** is a NumPy array with two extra abilities: it can live on a GPU,
and it can remember how it was computed so that gradients can be traced back through
it. We won't use the GPU at all; the second ability is the whole point of this
notebook.

For everyday use, tensors behave like arrays. The vocabulary carries straight over.

In [ ]:
t = torch.tensor([[1.0, 2.0, 3.0],
                  [4.0, 5.0, 6.0]])

print(t)
print("shape ", t.shape)        # same idea as .shape in NumPy
print("dtype ", t.dtype)
print()
print("t + 10  ->", (t + 10).tolist())
print("t * 2   ->", (t * 2).tolist())
print("t.sum() ->", t.sum().item())      # .item() pulls a single number out
print("t.mean()->", t.mean().item())

In [ ]:
# The NumPy operations you already know, with their torch equivalents.
a = torch.randn(4, 3)
b = torch.randn(3, 2)

print("a @ b        ", (a @ b).shape)            # matrix multiply, identical to NumPy
print("a.T          ", a.T.shape)                # transpose
print("a.reshape    ", a.reshape(3, 4).shape)
print("torch.relu   ", torch.relu(a).shape)
print("torch.sigmoid", torch.sigmoid(a).shape)
print("a.sum(dim=0) ", a.sum(dim=0).shape)       # NumPy calls this axis=, torch says dim=

### Converting back and forth

You will do this constantly: data arrives as NumPy, goes into the model as tensors,
and comes back out as NumPy for plotting.

In [ ]:
arr = np.array([[1.0, 2.0], [3.0, 4.0]])

t_from_np = torch.from_numpy(arr)       # shares memory with arr - no copy
back_to_np = t_from_np.numpy()

print("numpy -> tensor:", t_from_np.dtype)
print("tensor -> numpy:", type(back_to_np).__name__)

### Two gotchas worth meeting now

**1. dtype.** NumPy defaults to 64-bit floats; PyTorch defaults to **32-bit**. Mixing
them throws an error that looks alarming and means almost nothing:

> `RuntimeError: expected scalar type Double but found Float`

(The exact wording varies between PyTorch versions; the cell below prints whatever
yours says.) "Double" is float64, "Float" is float32. The fix is always the same: convert your
data with `.float()` or `torch.tensor(arr, dtype=torch.float32)`. Float32 is the
right default for neural networks — half the memory, and the extra precision buys
you nothing.

**2. `.detach()`.** A tensor that is part of a gradient computation refuses to be
turned into a NumPy array, because doing so would silently break the chain. Call
`.detach()` first to say "I only want the numbers".

In [ ]:
# The dtype error, on purpose.
f64 = torch.tensor([[1.0, 2.0]], dtype=torch.float64)
f32 = torch.tensor([[1.0], [2.0]], dtype=torch.float32)

try:
    f64 @ f32
except RuntimeError as e:
    print("RuntimeError:", e)

print()
print("after .float():", (f64.float() @ f32).item())

In [ ]:
# .detach(), and the error it prevents.
x = torch.tensor([2.0], requires_grad=True)
z = x * 3

try:
    z.numpy()
except RuntimeError as e:
    print("RuntimeError:", e)

print()
print("z.detach().numpy() ->", z.detach().numpy())

---
# 5.2 Autograd — the backward pass, for free

This is the feature that makes PyTorch worth using.

Mark a tensor with `requires_grad=True` and PyTorch starts recording every operation
you perform on it, building a graph of the computation as it goes. Call `.backward()`
on the final result, and it walks that graph in reverse — doing precisely the
blame-passing you wrote by hand in notebook 2 — and deposits the answer in each
tensor's `.grad`.

Start with something whose answer we already know. In notebook 1 we differentiated
`f(x) = x²` by hand and got `2x`, so at `x = 3` the gradient should be 6.

In [ ]:
x = torch.tensor(3.0, requires_grad=True)

f = x ** 2       # PyTorch quietly records: "f came from squaring x"
f.backward()     # walk the recording backwards

print("x      =", x.item())
print("f      =", f.item())
print("x.grad =", x.grad.item(), "   (2x = 6, as we derived by hand)")

Now something you would not want to differentiate by hand. PyTorch doesn't care how
tangled the expression is — it just follows the recording backwards.

In [ ]:
a = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(5.0, requires_grad=True)

out = torch.sigmoid(a * b) * torch.log(a ** 2 + b) + torch.tanh(a / b)
out.backward()

print("d(out)/da =", a.grad.item())
print("d(out)/db =", b.grad.item())

### The one thing that trips everyone up: gradients accumulate

`.backward()` **adds** to `.grad` rather than replacing it. That design is deliberate
(it lets you accumulate gradients over several batches before updating), but it means
that if you forget to clear them, every step is polluted by every step before it —
and your model trains badly for no visible reason.

Watch it happen.

In [ ]:
x = torch.tensor(3.0, requires_grad=True)

for i in range(3):
    f = x ** 2
    f.backward()
    print(f"backward call {i + 1}: x.grad = {x.grad.item()}")

print("\n^ 6, then 12, then 18. It is summing, not replacing.")

x.grad.zero_()      # clear it
f = x ** 2
f.backward()
print("after zero_():", x.grad.item())

**Rule:** clear the gradients at the top of every training step. In a moment we'll
use an optimizer, which gives us `optimizer.zero_grad()` for exactly this. Forgetting
that line is the most common bug in beginner PyTorch code — and since it doesn't
crash, you have to know to look for it.

---
# 5.3 Version 1: the notebook 2 network, with autograd

Here is our network built from raw tensors. The forward pass is a line-for-line
translation of the NumPy version. The backward pass — the fifteen lines of chain rule
we derived so carefully — is replaced by `loss.backward()`.

We deliberately start from **the same initial weights as the NumPy model**, so we can
compare gradients exactly.

In [ ]:
# Same starting parameters as np_params, now as tensors that track gradients.
# We keep float64 here purely so the comparison with NumPy is exact;
# real PyTorch code uses float32.
W1 = torch.tensor(np_params["W1"], requires_grad=True)
b1 = torch.tensor(np_params["b1"], requires_grad=True)
W2 = torch.tensor(np_params["W2"], requires_grad=True)
b2 = torch.tensor(np_params["b2"], requires_grad=True)

X_t = torch.tensor(X)          # float64, matching the parameters above
Y_t = torch.tensor(Y)

print("W1", tuple(W1.shape), W1.dtype, "requires_grad =", W1.requires_grad)
print("X_t", tuple(X_t.shape), X_t.dtype)

In [ ]:
import torch.nn.functional as F

# ---- forward pass: compare this with np_forward above ----
Z1 = X_t @ W1 + b1
A1 = torch.relu(Z1)
Z2 = A1 @ W2 + b2
A2 = torch.sigmoid(Z2)

loss = F.binary_cross_entropy(A2, Y_t)      # the same BCE we wrote by hand

print("loss =", loss.item())

# ---- backward pass: this one line replaces all of np_backward ----
loss.backward()

print("W1.grad", tuple(W1.grad.shape))
print("b1.grad", tuple(b1.grad.shape))
print("W2.grad", tuple(W2.grad.shape))
print("b2.grad", tuple(b2.grad.shape))

## The moment of truth

Our hand-derived gradients versus PyTorch's. If autograd is doing what we think it
is, these should be identical to floating-point precision.

In [ ]:
_, np_cache = np_forward(X, np_params)
np_grads = np_backward(X, Y, np_params, np_cache)

print(f"{'parameter':10} {'largest disagreement':>22}")
print("-" * 34)
for name, torch_param in [("W1", W1), ("b1", b1), ("W2", W2), ("b2", b2)]:
    diff = np.abs(torch_param.grad.numpy() - np_grads[name]).max()
    print(f"{name:10} {diff:22.3e}")

print()
print("Same numbers, to the last bit that floating point can represent.")
print("Autograd is not doing anything you did not already do by hand.")

That is the punchline of this notebook, and it's worth sitting with for a moment.

`loss.backward()` is not a black box that "does deep learning". It is the chain rule,
applied backwards through a recorded graph, arriving at exactly the numbers you
derived on paper in notebook 2. The difference between the two notebooks is that
PyTorch will do this for *any* architecture you invent, without you re-deriving
anything — which is the difference between a teaching exercise and a usable tool.

## Training with raw tensors

The loop is the same five steps as ever. Only step 3 got shorter.

In [ ]:
params_list = [W1, b1, W2, b2]
learning_rate = 0.5
losses_v1 = []

for step in range(4000):
    # 1. forward
    A2 = torch.sigmoid(torch.relu(X_t @ W1 + b1) @ W2 + b2)

    # 2. loss
    loss = F.binary_cross_entropy(A2, Y_t)
    losses_v1.append(loss.item())

    # 3. backward - clear first, because gradients accumulate
    for p in params_list:
        if p.grad is not None:
            p.grad.zero_()
    loss.backward()

    # 4. update. torch.no_grad() says "this arithmetic is not part of the model,
    #    do not record it" - without it, PyTorch would try to backprop through
    #    the update itself.
    with torch.no_grad():
        for p in params_list:
            p -= learning_rate * p.grad

print(f"loss  {losses_v1[0]:.4f}  ->  {losses_v1[-1]:.4f}")

with torch.no_grad():
    preds = torch.sigmoid(torch.relu(X_t @ W1 + b1) @ W2 + b2).numpy().ravel()
print(f"accuracy: {np.mean((preds > 0.5) == y):.1%}")

In [ ]:
def predict_v1(grid):
    with torch.no_grad():
        g = torch.tensor(grid, dtype=W1.dtype)
        return torch.sigmoid(torch.relu(g @ W1 + b1) @ W2 + b2).numpy().ravel()


fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(losses_v1, linewidth=2)
axes[0].set_xlabel("step"); axes[0].set_ylabel("loss"); axes[0].set_title("Loss")
plot_decision_boundary(predict_v1, X, y, title="Version 1: raw tensors + autograd",
                       ax=axes[1])
plt.tight_layout()
plt.show()

The same curve as notebook 2, from the same starting weights, by the same arithmetic
— with the backward pass written by PyTorch instead of by us.

---
# 5.4 Version 2: `nn.Linear`

Managing `W1, b1, W2, b2` by hand gets old quickly, and a ten-layer network would be
unbearable. `nn.Linear` bundles a weight matrix and a bias into one object that
initialises itself sensibly.

In [ ]:
layer = nn.Linear(in_features=2, out_features=16)

print("weight", tuple(layer.weight.shape))
print("bias  ", tuple(layer.bias.shape))
print()
print("both track gradients automatically:", layer.weight.requires_grad)

> **A shape surprise.** Our NumPy `W1` was `(2, 16)` — inputs by outputs — but
> `layer.weight` is `(16, 2)`, the other way round. That's because `nn.Linear`
> computes `x @ weight.T + bias` internally. It makes no difference to the maths or
> to you; it's just a convention to be aware of when you go poking at `.weight`
> directly.

Notice also that the layer arrived already initialised with small random numbers —
PyTorch applies a sensible scaling for you, for exactly the reason you proved in
notebook 2's zero-initialisation experiment.

In [ ]:
torch.manual_seed(0)

# float32 from here on - the normal PyTorch default.
X32 = torch.tensor(X, dtype=torch.float32)
Y32 = torch.tensor(Y, dtype=torch.float32)

lin1 = nn.Linear(2, 16)
lin2 = nn.Linear(16, 1)

params_v2 = list(lin1.parameters()) + list(lin2.parameters())
print("parameter tensors:", [tuple(p.shape) for p in params_v2])

losses_v2 = []
for step in range(4000):
    A2 = torch.sigmoid(lin2(torch.relu(lin1(X32))))     # forward
    loss = F.binary_cross_entropy(A2, Y32)              # loss
    losses_v2.append(loss.item())

    for p in params_v2:
        if p.grad is not None:
            p.grad.zero_()
    loss.backward()                                     # backward

    with torch.no_grad():
        for p in params_v2:
            p -= 0.5 * p.grad                           # update

with torch.no_grad():
    acc_v2 = ((torch.sigmoid(lin2(torch.relu(lin1(X32)))).numpy().ravel() > 0.5) == y).mean()
print(f"loss  {losses_v2[0]:.4f}  ->  {losses_v2[-1]:.4f}   accuracy {acc_v2:.1%}")

Same result, and we no longer create or initialise weights ourselves. But we are
still writing the update loop by hand, and that's the next thing to go.

---
# 5.5 Version 3: `nn.Sequential` and `torch.optim`

Two more pieces of machinery.

**`nn.Sequential`** chains layers together: the output of each becomes the input of
the next. When a model is a straight pipeline — as most simple ones are — this is all
the structure you need.

**An optimizer** owns the parameters and performs the update. `optim.SGD` does
exactly what we've been doing by hand (`p -= lr * p.grad`), but swapping it for
`optim.Adam` — which adapts the step size per parameter — is now a one-word change.

In [ ]:
import torch.optim as optim

torch.manual_seed(0)

model = nn.Sequential(
    nn.Linear(2, 16),
    nn.ReLU(),
    nn.Linear(16, 1),
    nn.Sigmoid(),
)

print(model)
print()
n_params = sum(p.numel() for p in model.parameters())
print("trainable parameters:", n_params)

Printing a model gives you its structure — a habit worth keeping, because it catches
wiring mistakes immediately.

`numel()` counts the numbers in a tensor, so that sum is the total parameter count:
our two layers hold 2×16 + 16 + 16×1 + 1 = 65 numbers. For scale, a small image model
has a few million, and a large language model has hundreds of billions. The training
loop below is, in outline, the same one used for all of them.

In [ ]:
loss_fn = nn.BCELoss()                                   # binary cross-entropy
optimizer = optim.SGD(model.parameters(), lr=0.5)        # the update rule

losses_v3 = []
for step in range(4000):
    optimizer.zero_grad()          # 1. clear last step's gradients
    y_hat = model(X32)             # 2. forward
    loss = loss_fn(y_hat, Y32)     # 3. loss
    loss.backward()                # 4. backward
    optimizer.step()               # 5. update

    losses_v3.append(loss.item())

print(f"loss  {losses_v3[0]:.4f}  ->  {losses_v3[-1]:.4f}")

**Those five lines are the canonical PyTorch training loop.** You will see them,
essentially unchanged, in every PyTorch project you ever open:

```python
optimizer.zero_grad()
output = model(batch)
loss = loss_fn(output, target)
loss.backward()
optimizer.step()
```

Compare them with the loop you wrote in notebook 1 — forward, loss, backward, update
— and note that nothing conceptual has changed. Only the amount of typing.

In [ ]:
def predict_torch(grid, model):
    """Run a NumPy grid through a torch model and get NumPy back."""
    model.eval()                                  # evaluation mode (matters later)
    with torch.no_grad():                         # don't record, we're not training
        g = torch.tensor(grid, dtype=torch.float32)
        out = model(g).numpy().ravel()
    model.train()
    return out


acc_v3 = ((predict_torch(X, model) > 0.5) == y).mean()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(losses_v1, label="v1 raw tensors", linewidth=2)
axes[0].plot(losses_v3, label="v3 Sequential + SGD", linewidth=2, linestyle="--")
axes[0].set_xlabel("step"); axes[0].set_ylabel("loss")
axes[0].set_title("Same network, same training")
axes[0].legend()

plot_decision_boundary(lambda g: predict_torch(g, model), X, y,
                       title=f"Version 3 - {acc_v3:.1%}", ax=axes[1])
plt.tight_layout()
plt.show()

(The two loss curves don't lie exactly on top of each other, and shouldn't: version 1
started from our NumPy He-initialised weights, version 3 from PyTorch's own random
initialisation. Different starting points, same destination.)

### Swapping the optimizer

Now that the optimizer is a separate object, trying a different one costs one line.
**Adam** keeps a running estimate of the right step size for each parameter
individually; it usually converges faster than plain SGD and needs less tuning, which
is why it is the default choice for most people most of the time.

In [ ]:
def train_quick(optimizer_name, lr, n_steps=4000, seed=0):
    torch.manual_seed(seed)
    m = nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 1), nn.Sigmoid())
    opt = {"SGD": optim.SGD, "Adam": optim.Adam}[optimizer_name](m.parameters(), lr=lr)
    fn = nn.BCELoss()
    hist = []
    for _ in range(n_steps):
        opt.zero_grad()
        loss = fn(m(X32), Y32)
        loss.backward()
        opt.step()
        hist.append(loss.item())
    return m, hist


plt.figure(figsize=(7.5, 4.5))
for name, lr in [("SGD", 0.5), ("SGD", 0.05), ("Adam", 0.01), ("Adam", 0.1)]:
    _, hist = train_quick(name, lr)
    plt.plot(hist, label=f"{name}, lr={lr}", linewidth=2)

plt.xlabel("step"); plt.ylabel("loss")
plt.title("SGD vs Adam")
plt.legend()
plt.show()

Adam typically drops faster in the early steps. Note that the useful learning rates
are quite different between the two — Adam's sweet spot is usually around `1e-3`,
SGD's is much larger. A learning rate is only meaningful relative to its optimizer,
so when you copy hyperparameters from a paper or a tutorial, copy the optimizer too.

---
# 5.6 Interlude: Python classes and inheritance

`nn.Sequential` handles pipelines, but real models often need something more flexible
— a branch, a skip connection, a bit of logic in the middle. For that, PyTorch wants
you to write a **class**. So here is the 5-minute refresher.

A **class** is a template that bundles data with the functions that operate on it. An
**object** (or *instance*) is one thing built from that template.

In [ ]:
class Dog:
    # __init__ runs once when you create an object. It sets up the data.
    # 'self' is the object being created; you never pass it in yourself.
    def __init__(self, name, age):
        self.name = name           # store on the object
        self.age = age

    # A method: a function that belongs to the object.
    def speak(self):
        return f"{self.name} says woof"


rex = Dog("Rex", 3)                # calls __init__
print(rex.name, "is", rex.age)
print(rex.speak())

**Inheritance** lets one class build on another. The child gets everything the parent
has, and can add to it or override parts of it.

`super().__init__(...)` calls the parent's setup. Forgetting it is a classic bug —
and in PyTorch it's a fatal one, because `nn.Module.__init__` is what sets up the
bookkeeping that tracks your parameters.

In [ ]:
class Animal:
    def __init__(self, name):
        self.name = name

    def speak(self):
        return f"{self.name} makes a sound"

    def describe(self):
        return f"I am {self.name}. {self.speak()}"


class Cat(Animal):                    # Cat inherits from Animal
    def __init__(self, name, indoor):
        super().__init__(name)        # run Animal's __init__ first
        self.indoor = indoor          # then add our own

    def speak(self):                  # override the parent's version
        return f"{self.name} says meow"


c = Cat("Mia", indoor=True)
print(c.describe())        # describe() came from Animal, but calls OUR speak()
print("indoor:", c.indoor)

Read that last line again, because it is the entire pattern PyTorch uses:

**`describe()` is defined in the parent, but when it calls `self.speak()` it gets the
child's version.** The parent provides the machinery; the child provides the specific
behaviour.

In PyTorch, `nn.Module` is the parent. It provides all the machinery — collecting
parameters, moving them between devices, saving and loading, switching between train
and eval mode. You provide one method, `forward()`, that says what your model
actually computes. When you call `model(x)`, the parent's machinery runs and calls
*your* `forward()`.

One more piece of Python to name: `__call__`. If a class defines it, its objects can
be called like functions. `nn.Module` defines `__call__` to do some housekeeping and
then invoke `forward()`. That is why you write `model(x)` and not `model.forward(x)`
— and you should always use `model(x)`, or the housekeeping gets skipped.

In [ ]:
class Doubler:
    def __init__(self, factor):
        self.factor = factor

    def __call__(self, x):            # makes the object callable
        return x * self.factor


double = Doubler(2)
print(double(21))        # calls __call__, no method name needed

---
# 5.7 Version 4: `nn.Module`

The idiomatic PyTorch model. Two methods:

- **`__init__`** — create the layers. Assigning them to `self` is what registers them,
  so `model.parameters()` finds them automatically.
- **`forward`** — say what happens to the input. Plain Python: you can branch, loop,
  print, or drop into a debugger.

This is the form you'll meet in essentially every PyTorch codebase, so it's worth
being comfortable with even though `nn.Sequential` would do for this model.

In [ ]:
class MoonsNet(nn.Module):
    def __init__(self, n_hidden=16):
        super().__init__()                          # never forget this line
        self.hidden = nn.Linear(2, n_hidden)
        self.output = nn.Linear(n_hidden, 1)

    def forward(self, x):
        x = self.hidden(x)
        x = torch.relu(x)
        x = self.output(x)
        return torch.sigmoid(x)


torch.manual_seed(0)
net = MoonsNet(n_hidden=16)

print(net)
print()
for name, p in net.named_parameters():
    print(f"  {name:16} {tuple(p.shape)}")

`named_parameters()` found all four tensors without being told about them. That
happened because we assigned the layers to `self` inside `__init__`, and
`nn.Module.__init__` — the line we were careful not to forget — installed the
bookkeeping that notices such assignments.

**🔬 Try it.** Comment out `super().__init__()` and re-run. You get a clear error
telling you the module wasn't initialised. Now imagine debugging that without knowing
what `super()` does — which is why we did the refresher first.

In [ ]:
torch.manual_seed(0)
net = MoonsNet(n_hidden=16)

loss_fn = nn.BCELoss()
optimizer = optim.Adam(net.parameters(), lr=0.01)

losses_v4 = []
for step in range(2000):
    optimizer.zero_grad()
    loss = loss_fn(net(X32), Y32)
    loss.backward()
    optimizer.step()
    losses_v4.append(loss.item())

    if step % 500 == 0:
        print(f"step {step:5}  loss {loss.item():.4f}")

acc_v4 = ((predict_torch(X, net) > 0.5) == y).mean()
print(f"\nfinal loss {losses_v4[-1]:.4f}   accuracy {acc_v4:.1%}")

plot_decision_boundary(lambda g: predict_torch(g, net), X, y,
                       title=f"Version 4: nn.Module + Adam - {acc_v4:.1%}")
plt.show()

## A note on `BCEWithLogitsLoss`

Our models end with `nn.Sigmoid()`, and we pair that with `nn.BCELoss()`. It's the
clearest way to write it, and it matches notebook 2 exactly, which is why we've done
it that way so far.

In practice you'll usually see something slightly different: the model ends **without**
a sigmoid, outputting raw scores (called **logits**), and the loss is
`nn.BCEWithLogitsLoss()`, which applies the sigmoid internally.

Why bother? Because the combined version cancels some terms algebraically before
computing anything, which avoids `log(0)` — the same `NaN` risk we papered over with
`np.clip` in notebook 1. Mathematically identical, numerically much better behaved.

The trap is that it's easy to end up applying sigmoid twice: once in your model and
once inside the loss. The model then trains, badly, without complaining. **If you use
`BCEWithLogitsLoss`, your model must not end in a sigmoid** — and you must remember
to apply `torch.sigmoid` yourself when you want probabilities out.

In [ ]:
class MoonsNetLogits(nn.Module):
    """Same network, but outputs raw scores instead of probabilities."""

    def __init__(self, n_hidden=16):
        super().__init__()
        self.hidden = nn.Linear(2, n_hidden)
        self.output = nn.Linear(n_hidden, 1)

    def forward(self, x):
        return self.output(torch.relu(self.hidden(x)))     # no sigmoid


torch.manual_seed(0)
net_logits = MoonsNetLogits()
loss_fn = nn.BCEWithLogitsLoss()                            # sigmoid lives in here
optimizer = optim.Adam(net_logits.parameters(), lr=0.01)

for step in range(2000):
    optimizer.zero_grad()
    loss = loss_fn(net_logits(X32), Y32)
    loss.backward()
    optimizer.step()


def predict_logits(grid):
    with torch.no_grad():
        g = torch.tensor(grid, dtype=torch.float32)
        return torch.sigmoid(net_logits(g)).numpy().ravel()   # sigmoid by hand


acc_logits = ((predict_logits(X) > 0.5) == y).mean()
print(f"BCEWithLogitsLoss version - accuracy {acc_logits:.1%}, loss {loss.item():.4f}")

Same answer, better numerics. We'll use the logits style in notebook 4, where the
multi-class equivalent (`nn.CrossEntropyLoss`) works exactly the same way.

---
# 5.8 Side by side

Everything you built by hand, and its PyTorch equivalent:

| What it does | NumPy (notebooks 1–2) | PyTorch |
|---|---|---|
| Array of numbers | `np.array(...)` | `torch.tensor(...)` |
| Matrix multiply | `A @ B` | `A @ B` |
| Activation | `np.maximum(0, z)` | `torch.relu(z)` |
| Layer | `W = rng.normal(...)`, `X @ W + b` | `nn.Linear(in, out)` |
| Loss | `bce_loss(y_hat, y)` | `nn.BCELoss()` |
| **Backward pass** | **~15 lines of chain rule** | **`loss.backward()`** |
| Clear gradients | (nothing — you rebuilt them) | `optimizer.zero_grad()` |
| Update | `p -= lr * grad` for each `p` | `optimizer.step()` |
| Stack of layers | write it out | `nn.Sequential(...)` |
| Custom model | a dict of arrays | subclass `nn.Module` |

And the training loop, which is the same shape in both:

```python
#   NumPy                              PyTorch
#   ------------------------------     ------------------------------
#                                      optimizer.zero_grad()
#   y_hat, cache = forward(X, p)       y_hat = model(X)
#   loss = bce_loss(y_hat, Y)          loss = loss_fn(y_hat, Y)
#   grads = backward(X, Y, p, cache)   loss.backward()
#   for k in p: p[k] -= lr*grads[k]    optimizer.step()
```

PyTorch didn't replace your understanding. It replaced your typing.

---
# Recap

- **Tensors** — NumPy arrays that can track gradients. Watch the float32/float64
  boundary, and use `.detach()` to get plain numbers out.
- **Autograd** — `requires_grad=True` records the computation, `.backward()` replays
  it in reverse. We checked its output against our hand-derived gradients and they
  matched exactly.
- **Gradients accumulate** — clear them every step, or train badly and never know why.
- **`nn.Linear`** — a weight matrix and a bias, initialised for you. Note the
  transposed `.weight` shape.
- **`nn.Sequential` + `torch.optim`** — the five-line training loop you'll see
  everywhere.
- **`nn.Module`** — `__init__` builds the layers, `forward` uses them, `super()` is
  not optional.
- **`BCEWithLogitsLoss`** — no sigmoid in the model, better numerics.

### Check yourself

1. You train a model and the loss barely moves. You notice there's no
   `optimizer.zero_grad()` in your loop. What was going wrong?
2. Why does `.numpy()` fail on a tensor that has `requires_grad=True`?
3. What breaks if you forget `super().__init__()` in an `nn.Module` subclass?
4. Your model ends with `nn.Sigmoid()` and you switch the loss to
   `nn.BCEWithLogitsLoss()`. What now happens, and would it raise an error?
5. Which lines of the five-line training loop correspond to which lines of the NumPy
   loop from notebook 2?

### Optional stretch

- Add a second hidden layer to `MoonsNet`. Two lines changed, versus the substantial
  surgery it would take in notebook 2's code.
- Replace `torch.relu` with `nn.LeakyReLU()` and see whether it helps the narrow
  networks (`n_hidden=3`) that struggled in notebook 2.
- Save and reload a model: `torch.save(net.state_dict(), "net.pt")`, then
  `net2 = MoonsNet(); net2.load_state_dict(torch.load("net.pt"))`.
- Print `net.hidden.weight.grad` after a `backward()` call and confirm it has the same
  shape as `net.hidden.weight`.

### Next

**Notebook 4 — MNIST and a small CNN.** A real dataset: 70,000 handwritten digits.
We meet mini-batches, `DataLoader`, the train/test split that finally lets us *see*
overfitting, and convolutional layers.